In [1]:
# Cell 1 – Imports & basic settings
import requests          # HTTP client
import pandas as pd      # Data handling / tabular view
import json              # Optional pretty‑print

# Optional: make notebooks display DataFrames nicely
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

In [2]:
# Cell 2 – Define the API endpoint
url = (
    "https://api.jimbogo.com/api/jimbo/v1.1/published-content-objects-search/"
    "?start=0&count=300&alias=jimbogo&site=jimbogo&organisationId=0c20e485-4e27-4d6f-9d33-bcb778c701e6"
)

# If the API requires a header (e.g., Accept: application/json), add it here:
headers = {
    "Accept": "application/json",
    # "Authorization": "Bearer <your‑token>"   # ← uncomment if auth is needed
}

In [3]:
# Cell 3 – Fetch the JSON payload
response = requests.get(url, headers=headers, timeout=30)

# Raise an informative error if something went wrong
response.raise_for_status()          # will raise HTTPError for non‑2xx codes

# Parse the JSON text into a Python dict/list
data = response.json()
#print(f"Top‑level keys: {list(data.keys())}")

In [4]:
# Cell 4 – Inspect a small sample (optional)
# Many APIs wrap the actual records inside a field like "items" or "results".
# Adjust the key name according to what you see in the output above.
sample_key = "objects"   # <-- change if your payload uses a different name
if sample_key in data:
    records = data[sample_key]
else:
    # Fallback: assume the whole payload is a list of objects
    records = data

print(f"Number of records retrieved: {len(records)}")
#print(records[:2])   # show first two objects for a quick sanity check

Number of records retrieved: 176


In [5]:
# Cell 5 – Normalise into a pandas DataFrame
# This works well when each record is a flat dict or when you want to flatten nested dicts.
df = pd.json_normalize(records)

# Quick look at the resulting table
#display(df.head())
#print("\nColumns discovered:")
#print(df.columns.tolist())

In [6]:
# Cell 6 – Example: keep only the fields you care about
# Replace the column names with whatever is relevant for your analysis.
columns_of_interest = [
    "instance.id",
    "instance.name",
    "instance.contentAttributes.routeEncodedPolyLines"
    # add more keys as needed …
]

# Some columns may be missing in some rows; pandas will fill with NaN.
df_selected = df[columns_of_interest].copy()
#display(df_selected.head())

In [7]:
df_selected = df_selected.rename(columns={'instance.id': 'klompenpad_id','instance.name': 'klompenpad_naam', 'instance.contentAttributes.routeEncodedPolyLines': 'routeEncodedPolyLines'})

In [8]:
df_selected['klompenpad_naam'] = df_selected['klompenpad_naam'].str.replace(' ', '-', regex=False)

In [9]:
df_selected['url'] = 'https://klompenpaden.nl/klompenpaden/' + df_selected['klompenpad_naam'].astype(str) + '/' + df_selected['klompenpad_id'].astype(str)

In [10]:
df_csv = pd.read_csv('klompenpaden_runs.csv', sep=';')

In [11]:
# Specify join type
df_selected = pd.merge(df_selected, df_csv, on='klompenpad_naam', how='left')  # inner, outer, left, right

In [12]:
import polyline

In [13]:
def parse_list_string(s):
    """Convert string representation of list to actual list"""
    if pd.isna(s) or not isinstance(s, str):
        return []
    try:
        # Safely evaluate the string as a Python literal
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        print(f"Could not parse: {s[:50]}...")
        return []

# Convert if necessary
if isinstance(df_selected['routeEncodedPolyLines'].iloc[0], str):
    df_selected['polyline_lists'] = df_selected['routeEncodedPolyLines'].apply(parse_list_string)
else:
    df_selected['polyline_lists'] = df_selected['routeEncodedPolyLines']

In [14]:
def decode_polyline_list(polyline_list):
    """Decode a list of polyline strings"""
    if not polyline_list:
        return []
    
    all_coords = []
    for encoded in polyline_list:
        try:
            decoded = polyline.decode(encoded)
            all_coords.extend(decoded)
        except Exception as e:
            print(f"Warning: Failed to decode '{encoded[:20]}...': {e}")
            continue
    
    return all_coords

# Apply decoding
df_selected['decoded_coordinates'] = df_selected['polyline_lists'].apply(decode_polyline_list)

# View results
#print(df_selected[['routeEncodedPolyLines', 'decoded_coordinates']].head())

In [15]:
import geopandas as gpd
from shapely.geometry import LineString

# Create GeoDataFrame
geometry = [LineString([(lon, lat) for lat, lon in coords]) 
            for coords in df_selected['decoded_coordinates'] if coords]

gdf = gpd.GeoDataFrame(
    df_selected[df_selected['decoded_coordinates'].apply(lambda x: x is not None)],
    geometry=geometry,
    crs="EPSG:4326"  # WGS84 coordinate reference system
)

In [16]:
# Drop all columns except geometry and specific ones
keep_cols = ['klompenpad_naam', 'url', 'date', 'geometry']
gdf = gdf[[col for col in gdf.columns if col in keep_cols]]

In [17]:
# Save to GeoJSON
gdf.to_file('klompenpaden_routes.geojson', driver='GeoJSON')
print(f"Saved {len(gdf)} routes to klompenpaden_routes.geojson")

Saved 176 routes to klompenpaden_routes.geojson
